In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

/Users/admin/repos/conferease


In [2]:
import cv2
import mediapipe as mp
import numpy as np
import cv2
import mediapipe as mp
import numpy as np



def convert_to_3d(landmarks, image_width, image_height, scale_factor=1):
    landmarks_3d = []
    for landmark in landmarks:
        x = landmark.x * image_width * scale_factor
        y = (1 - landmark.y) * image_height * scale_factor  # Invert Y
        z = landmark.z * scale_factor
        landmarks_3d.append([x, y, z])
    return landmarks_3d

def adjust_hand_coordinates(hand_landmarks, wrist_landmark, image_width, image_height):
    adjusted_landmarks = convert_to_3d(hand_landmarks, 1, 1)
    
    for i in range(len(adjusted_landmarks)):
        # Adjust Z-coordinate relative to the wrist
        adjusted_landmarks[i][2] /= 0.4
        adjusted_landmarks[i][2] += wrist_landmark[2]

        adjusted_landmarks[i][1] = wrist_landmark[1] + adjusted_landmarks[i][1] * 0.4
        adjusted_landmarks[i][0] = (wrist_landmark[0] + adjusted_landmarks[i][0]) * 0.4 + 0.1
    return adjusted_landmarks

def process_frame_holistic(frame, mp_holistic, image_width, image_height):
    # Convert the image to RGB
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Process the image with holistic model
    results = mp_holistic.process(image)
    
    frame_data = {}
    
    if results.pose_landmarks:
        # Convert pose landmarks to 3D coordinates
        frame_data['pose'] = convert_to_3d(results.pose_landmarks.landmark, 1, 1)
    
        left_wrist = frame_data['pose'][15]  # Left wrist
        right_wrist = frame_data['pose'][16]  # Right wrist
        
        # Process hand landmarks
        hand_landmarks = []
        if results.left_hand_landmarks:
            left_hand = adjust_hand_coordinates(results.left_hand_landmarks.landmark, left_wrist, image_width, image_height)
            hand_landmarks.append(left_hand)
        if results.right_hand_landmarks:
            right_hand = adjust_hand_coordinates(results.right_hand_landmarks.landmark, right_wrist, image_width, image_height)
            hand_landmarks.append(right_hand)
        
        if hand_landmarks:
            frame_data['hands'] = hand_landmarks
    
    return frame_data

def process_video(video_path, num_frames=10):
    # Open the video file
    cap = cv2.VideoCapture(video_path)

    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"FPS: {fps}, Total Frames: {frame_count}, Width: {frame_width}, Height: {frame_height}")

    # Calculate frame indices for num_frames evenly spaced frames
    frame_indices = np.linspace(0, frame_count - 1, num_frames, dtype=int)

    animation_data = {
        "frame_rate": fps,
        "keyframes": []
    }
    
    mp_holistic = mp.solutions.holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)
    
    all_coords = {'x': [], 'y': [], 'z': []}
    
    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
        ret, frame = cap.read()
        if not ret:
            print(f"Failed to read frame {frame_index}")
            continue
        
        frame_data = process_frame_holistic(frame, mp_holistic, frame_width, frame_height)

        # Collect coordinates for bounds calculation
        if 'pose' in frame_data:
            for coord in frame_data['pose']:
                all_coords['x'].append(coord[0])
                all_coords['y'].append(coord[1])
                all_coords['z'].append(coord[2])
        if 'hands' in frame_data:
            for hand in frame_data['hands']:
                for coord in hand:
                    all_coords['x'].append(coord[0])
                    all_coords['y'].append(coord[1])
                    all_coords['z'].append(coord[2])
                    
        # Add to animation data
        animation_data["keyframes"].append({
            "frame": int(frame_index),
            "landmarks": frame_data
        })

    cap.release()
    
    # Calculate bounds
    x_min, x_max = min(all_coords['x']), max(all_coords['x'])
    y_min, y_max = min(all_coords['y']), max(all_coords['y'])
    z_min, z_max = min(all_coords['z']), max(all_coords['z'])
    
    animation_data['bounds'] = {
        'x': [x_min, x_max],
        'y': [y_min, y_max],
        'z': [z_min, z_max]
    }
    
    return animation_data

video_path = 'WLASL/start_kit/raw_videos/05727.mp4'
animation_data = process_video(video_path, num_frames=80)
app = QApplication([])
ex = LandmarkVisualizerApp(animation_data, video_path)
ex.show()
app.exec_()

FPS: 29.97002997002997, Total Frames: 87, Width: 1280, Height: 720


I0000 00:00:1719584024.644885 17317236 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1719584024.748654 17319617 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1719584024.755995 17319618 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1719584024.757283 17319618 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1719584024.757371 17319623 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1719584024.757633 17319622 inference_feedback_manager.cc:114] Feedback manager requires